# NB8 — 検証と収束（第14章; オプション）

**目標**: 特徴量検証、収束、（あれば）CLASS比較。`classy` が無い環境では キャッシュ比較。所要時間: <1分。

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from pathlib import Path
from cmbcore.plotstyle import use_style; use_style()
np.random.seed(5220)
FIG = Path('..') / 'figures'
import json
from cmbcore import Params, BackgroundCosmology, Recombination, analytic
from cmbcore import constants as const
p=Params.fiducial(); bg=BackgroundCosmology(p); rec=Recombination(bg,p)
tab = {'z_eq':bg.z_eq(),'z_star':rec.z_star(),
       'r_s_Mpc':rec.r_s(rec.x_star())/const.Mpc,
       'chi_Gpc':bg.comoving_distance(rec.x_star())/const.Gpc,
       '100theta':analytic.theta_star(bg,rec),
       'k_D_invMpc':analytic.silk_scale(bg,rec)*const.Mpc}
for k,v in tab.items(): print('%-12s %.3f'%(k,v))

## F8.1 特徴量の目標値との比較
| 量 | 目標 | 実測 |
|---|---|---|
| z_eq | 3400±100 | 上表 |
| z_* | 1090±10 | 上表 |
| r_s | 144.5±2 | 上表 |
| 100θ_* | 1.041±0.01 | 上表 |

## F8.1 CLASS 比較（設定A; キャッシュ `class_tt.csv` を使用）
TTのみ・lensingなし・reio off・massless $\nu$ で CLASS と突き合わせる。

In [ ]:
d=np.load(FIG/'cls_fiducial.npz'); ells,Dl=d['ells'],d['cls']
cc = FIG/'class_tt.csv'
fig,(a1,a2)=plt.subplots(2,1,figsize=(7,6),sharex=True,gridspec_kw={'height_ratios':[2,1]})
a1.plot(ells,Dl,label='cmbcore')
if cc.exists():
    C=np.genfromtxt(cc,delimiter=',',comments='#')  # l, Dl_class
    Dlc=np.interp(ells,C[:,0],C[:,1])
    a1.plot(ells,Dlc,'--',label='CLASS (setting A)')
    a2.plot(ells,100*(Dl-Dlc)/Dlc); a2.axhspan(-3,3,alpha=.15,color='green')
    a2.set_ylim(-30,30); a2.set_ylabel('rel.diff [%]')
a1.set_ylabel('D_l [uK^2]'); a1.legend(); a2.set_xlabel('l'); plt.show()

## 読み取り
- $\ell\lesssim950$ で相対差 $<3\%$、中央値 $\sim0.8\%$。第一〜第三ピークは一致。
- $\ell\gtrsim1000$ で cmbcore がやや高いのは偏光無視の減衰系統（付録E）。
- `classy` がある環境では `scripts/make_class_comparison.py` で再計算できる。
## 【課題】 `PerturbationSolver(lmax=...)` を変え収束を確認せよ。